# Why Transform Queries? [Step 1 - The Vocabulary Mismatch Problem]

> **MLCourse - Agentic AI - Advanced RAG - Query Transformation**

Every retrieval technique in this track so far has taken the user's question as
given, and worked on the *index* side: better chunks, better fusion, better
reranking. This module works on the other side of the comparison - **the query
itself**.

The reason is a failure mode as old as information retrieval, called the
**vocabulary mismatch problem**: the words a user chooses to ask a question are
usually *not* the words the answer is written in. Documents say "the accused",
users say "the guy on trial". Documents say "hypotension", users say "low blood
pressure". No amount of reranking helps, because the right document was never
retrieved.

This notebook demonstrates the problem concretely, shows why embeddings only
partly solve it, and lays out the family of fixes the rest of the module builds.

### 1. Setup

`GROQ_API_KEY` is loaded from `03_agentic_ai/.env` by walking up the directory
tree, so this notebook runs from any working directory.

In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


We use **Groq** with `qwen/qwen3.8-27b`. The free tier is around **8000 tokens
per minute**, and every transformation technique in this module spends tokens
*before* retrieval even starts - so `ask()` below paces itself and backs off
exponentially on failure.

A local Ollama server at `http://localhost:11434` is the documented fallback;
swapping `ChatGroq` for `ChatOllama(model="llama3.1:8b")` is the only change
needed.

In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


### 2. Two ways a query can fail

Retrieval compares a query to documents. That comparison can fail for two
distinct reasons, and they need different fixes:

**Lexical mismatch** - the user's words never appear in the document. BM25 sees
zero overlap and scores it zero. This is what dense embeddings were invented to
fix, and they fix it well.

**Asymmetry** - a deeper problem that embeddings do *not* fix. A question and
its answer are different kinds of text. "Why was the rabbit hurrying?" is short,
interrogative, and abstract. The passage that answers it is long, narrative, and
concrete: *"Oh dear! Oh dear! I shall be too late!"*. Embedding models place
text near other text that *looks like it* - and a question looks like other
questions, not like its answer.

That asymmetry is the thing query transformation attacks.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)


def embed(text):
    return encoder.encode([text], normalize_embeddings=True)[0]


def dense_rank(text, top_n=10):
    """Rank paragraph indices by cosine similarity to `text` (best first)."""
    sims = doc_vectors @ embed(text)
    return [int(i) for i in np.argsort(sims)[::-1][:top_n]]


def dense_scores(text):
    return doc_vectors @ embed(text)


print("dense index ready:", doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense index ready: (237, 384)


In [5]:
# A question, and the exact passage that answers it, embedded independently.
question = "Why was the White Rabbit in such a hurry?"

answer_style = ("The Rabbit took a watch out of its waistcoat-pocket and said "
                "'Oh dear! Oh dear! I shall be too late!' and hurried on.")
question_style = "What is the reason that the rabbit was rushing and anxious?"
unrelated = "The garden was full of bright flower-beds and cool fountains."

qv = embed(question)
for label, text in [("answer-shaped text ", answer_style),
                    ("question-shaped text", question_style),
                    ("unrelated text     ", unrelated)]:
    print(f"cos={float(qv @ embed(text)):.3f}   {label}   {text[:70]}...")

cos=0.580   answer-shaped text    The Rabbit took a watch out of its waistcoat-pocket and said 'Oh dear!...
cos=0.771   question-shaped text   What is the reason that the rabbit was rushing and anxious?...
cos=0.169   unrelated text        The garden was full of bright flower-beds and cool fountains....


Read that carefully. The **question-shaped** rephrasing - which contains none of
the answer - typically scores *higher* than the passage that actually answers
the question.

The embedding model is not broken. It is doing exactly what it was trained to
do: place semantically similar text nearby. It is just that "similar to a
question" and "answers a question" are different relations, and only one of them
is what we want.

### 3. The mismatch, measured on real retrieval

Let us take a question and three progressively more "document-like" rewrites of
it, and see what each retrieves from the corpus.

In [6]:
# An evaluation question is only useful if we can decide, mechanically and
# without an LLM, whether a retrieved paragraph is relevant. We do that with
# required keyword sets: a paragraph counts as relevant when it contains every
# keyword in at least one of the "any_of" groups. This is a strict, honest,
# reproducible judgement - no LLM grading, no hand-waving.

EVAL_QUESTIONS = [
    {"q": "Why was the White Rabbit in such a hurry?",
     "any_of": [["rabbit", "hurry"], ["rabbit", "late"], ["oh dear", "late"]]},
    {"q": "What happened when Alice drank from the little bottle?",
     "any_of": [["drink", "bottle"], ["bottle", "shutting up like a telescope"],
                ["drank", "telescope"]]},
    {"q": "What game does the Queen of Hearts make everyone play?",
     "any_of": [["croquet"], ["flamingo", "hedgehog"]]},
    {"q": "Who does Alice meet at the mad tea party?",
     "any_of": [["hatter", "dormouse"], ["march hare", "hatter"], ["tea", "dormouse"]]},
    {"q": "What advice does the Caterpillar give Alice?",
     "any_of": [["caterpillar", "mushroom"], ["caterpillar", "keep your temper"],
                ["caterpillar", "who are you"]]},
    {"q": "How does the Cheshire Cat disappear?",
     "any_of": [["grin", "vanish"], ["cheshire cat", "grin"], ["vanished", "grin"]]},
    {"q": "What does the Queen shout whenever she is angry?",
     "any_of": [["off with"], ["queen", "executed"]]},
    {"q": "What happens at the trial of the Knave of Hearts?",
     "any_of": [["knave", "tarts"], ["jury", "verdict"], ["sentence", "verdict"]]},
]


def is_relevant(doc_text, question):
    """True when the paragraph satisfies any one keyword group for the question."""
    low = doc_text.lower()
    return any(all(word in low for word in group) for group in question["any_of"])


def precision_at_k(ranked_ids, question, k=5):
    """Fraction of the top-k retrieved paragraphs that are relevant."""
    top = ranked_ids[:k]
    return sum(is_relevant(paragraphs[i], question) for i in top) / max(len(top), 1)


# Sanity check: every question must have at least one relevant paragraph in
# the corpus, otherwise the metric is meaningless.
for question in EVAL_QUESTIONS:
    n_rel = sum(is_relevant(p, question) for p in paragraphs)
    print(f"{n_rel:3d} relevant paragraphs | {question['q']}")

  7 relevant paragraphs | Why was the White Rabbit in such a hurry?
  3 relevant paragraphs | What happened when Alice drank from the little bottle?
  9 relevant paragraphs | What game does the Queen of Hearts make everyone play?
 10 relevant paragraphs | Who does Alice meet at the mad tea party?
  2 relevant paragraphs | What advice does the Caterpillar give Alice?
  1 relevant paragraphs | How does the Cheshire Cat disappear?
  5 relevant paragraphs | What does the Queen shout whenever she is angry?
  1 relevant paragraphs | What happens at the trial of the Knave of Hearts?


In [7]:
variants = {
    "user question  ": "Why was the White Rabbit in such a hurry?",
    "keywords only  ": "white rabbit hurry late watch",
    "answer-shaped  ": "The White Rabbit pulled a watch from his waistcoat pocket, "
                       "cried that he would be too late, and hurried away.",
}

target = EVAL_QUESTIONS[0]        # the White Rabbit question, with its keyword rules

for label, text in variants.items():
    ranked = dense_rank(text, top_n=5)
    p5 = precision_at_k(ranked, target, 5)
    hit = next((r for r, d in enumerate(ranked, 1)
                if is_relevant(paragraphs[d], target)), None)
    print(f"{label} P@5={p5:.2f}  first relevant at rank {hit}")
    print(f"                 top hit: {paragraphs[ranked[0]][:95]}...")
    print()

user question   P@5=0.40  first relevant at rank 1
                 top hit: After a time she heard a little pattering of feet in the distance, and she hastily dried her ey...

keywords only   P@5=0.60  first relevant at rank 1
                 top hit: Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it w...

answer-shaped   P@5=0.60  first relevant at rank 1
                 top hit: After a time she heard a little pattering of feet in the distance, and she hastily dried her ey...



The answer-shaped variant almost always retrieves better than the raw question,
even though it is *longer* and contains the same information. That single
observation is the seed of the whole module - and specifically of **HyDE**
(notebook 02), which asks an LLM to write that answer-shaped text for you.

### 4. The four transformation families

There are four broadly useful moves, and they fix different failures. Each gets
its own notebook.

| technique | the move | fixes | notebook |
|---|---|---|---|
| **HyDE** | write a hypothetical *answer*, embed that instead of the question | question/answer asymmetry | [02](02_hyde.ipynb) |
| **Multi-query expansion** | generate several rephrasings, retrieve with all, fuse | one phrasing is unlucky | [03](03_multi_query_expansion.ipynb) |
| **Step-back prompting** | ask a broader question first, retrieve background too | question too specific for the index | [04](04_step_back_prompting.ipynb) |
| **Decomposition** | split a multi-part question into sub-questions | question needs several documents | [`../03_agentic_rag/03_query_decomposition.ipynb`](../03_agentic_rag/03_query_decomposition.ipynb) |

**A note on decomposition.** It belongs to this family, but this course already
teaches it properly in
[`../03_agentic_rag/03_query_decomposition.ipynb`](../03_agentic_rag/03_query_decomposition.ipynb),
where an agent splits a compound question and retrieves per part. We deliberately
do **not** repeat it here. When you finish this module, go read that notebook -
it is the fifth technique, taught in its natural agentic setting.

### 5. A first taste: ask the model to diagnose the query

Before generating rewrites, it helps to see that an LLM can *recognise* the
mismatch. Here we ask Groq to say what vocabulary the answer would likely use.

In [8]:
diagnosis = ask(
    "A user searching a 19th-century novel typed this query:\n\n"
    "  'Why was the White Rabbit in such a hurry?'\n\n"
    "The search engine compares the query text to passages of the novel. "
    "In 3-4 sentences, explain what words and phrases the actual answering "
    "passage is likely to contain that this query does NOT contain, and why "
    "that hurts retrieval."
)
print(diagnosis)

The actual passage likely contains specific proper nouns and descriptive details, such as "Alice," "pocket-watch," "late," or "important appointment," which are absent from the user's generic query. Since the query relies on the common noun "White Rabbit" and the vague concept of "hurry," it lacks the unique lexical overlap needed to distinguish this specific scene from other mentions of the character or general descriptions of haste. This mismatch reduces the similarity score between the query and the relevant text, causing the search engine to potentially rank less relevant passages higher or miss the correct answer entirely due to insufficient keyword matching.


### 6. The cost you are accepting

Query transformation is not free, and the costs are different in kind from
reranking:

- **Latency before retrieval.** An LLM call now sits on the critical path
  *before* you can even start searching. That is typically 300-1500 ms.
- **Token spend per query.** On Groq's 8000 tokens/minute free tier, multi-query
  expansion with five variants is a meaningful fraction of your budget.
- **A new failure mode.** The LLM can hallucinate a rewrite that drags retrieval
  somewhere wrong. Unlike reranking - which can only reorder what you already
  found - a bad transformation can make results *worse* than the raw query.
- **Cacheability.** Rewrites are deterministic-ish at `temperature=0`, so a
  query-string cache removes most of the cost for repeated queries. Use one.

Notebook 05 measures all of this against a fixed question set instead of
guessing.

### 7. Key takeaways

- **Vocabulary mismatch**: users and documents use different words, so a literal
  query-to-document comparison under-retrieves.
- Dense embeddings solve *lexical* mismatch but not **question/answer
  asymmetry** - a question embeds near other questions.
- Answer-shaped query text retrieves better than question-shaped text, and you
  can measure this on your own corpus in a few lines.
- Four families of fix: HyDE, multi-query expansion, step-back prompting, and
  decomposition (taught in `../03_agentic_rag`).
- Every transformation costs an LLM call before retrieval and can make things
  worse - so measure, and cache.

Next: [`02_hyde.ipynb`](02_hyde.ipynb) - write the answer first, then search
with it.